https://docs.sqlalchemy.org/en/20/tutorial/index.html#unified-tutorial

# SQLAlchemy Unified Tutorial

The SQLAlchemy Unified Tutorial is integrated between the Core and ORM components of SQLAlchemy and serves as a unified introduction to SQLAlchemy as a whole. For users of SQLAlchemy within the 1.x series, in the [2.0 style](https://docs.sqlalchemy.org/en/20/glossary.html#term-2.0-style) of working, the ORM uses Core-style querying with the [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) construct, and transactional semantics between Core connections and ORM sessions are equivalent. Tke note of the blue border styles for each section, that will tell you how "ORM-ish" a particular topic is!

Users who are already familiar with SQL Alchemy, and especially those looking to migrate existing applications to work under SQLAlchemy 2.0 series within the 14 transitional phase should check out the [SQLAlchemy 2.0 - Major Migration Guide](https://docs.sqlalchemy.org/en/20/changelog/migration_20.html) document as well.

For the newcomer, this document has a **lot** of detail, however by the end they will be considered an **Alchemist**

SQLAlchemy is presented as two distinct APIs, one building on top of the other. These APIs are known as **Core** and **ORM**.

**SQLAlchemy Core** is the foundational architecture for SQLAlchemy as a "database toolkit". The library provides tools for managing connectivity to a database, interacting with database queries and results, and programmatic construction of SQL statements.

Sections that are **primarily Core-only** will not refer to the ORM. SQLAlchemy constructs used in these sections will be imported from the `sqlalchemy` namespace. As an additional indicator of subject classification, they will also include a **dark blue border on the right**. When using the ORM, these concept are still in play but are less often explicit in user code. ORM users should read these sections, but not expect to be using these APIs directly for ORM-centric code.


**SQLAlchemy ORM** builds upon the Core to provide optional **object relational mapping** capabilities. The ORM provides an additional configuration layer allowing user-defined Python classes to be **mapped** to database tables and other constructs, as well as an object persistence mechanism known as the **Session**. It then extends the Core-level SQL Expression Language to allow SQL queries to be composed and invoked in terms of user-defined objects.

Sections that are **primarily ORM-only** should be **titled to include the phrase 'ORM'**, so that it's clear this is an ORM related topic. SQLAlchemy constructs used in these sections will be imported from the `sqlalchemy.orm` namespace. Finally, as an additional indicator of subject classification, they will also include a **light blue border on the left**. Core-only users can skip these.

**Most** sections in this tutorial discuss **Core concepts that are also used explicitly with the ORM**. SQLAlchemy 2.0 in particular features a much greater level of integration of Core API use within the ORM. For each of these sections, there will be **introductory text** discussing the degree to which ORM users should expect to be using these prorgamming patterns. SQLAlchmey constructs in these sections will be imported from the `sqlalchemy` namespace with some potential use of `sqlalchemy.orm` constructs at the same time. As an additional indicator of subject classification, these sections will also include **both a thinner light border on the left, and a thicker dark border on the right**. Core and ORM users should familiarize with concepts in these sections equally.


# Tutorial Overview

The tutorial will present both concepts in the natural order that they should be learned, first with a mostly-Core-centric approach and then spanning out into more ORM-centric concepts. The major sections of this tutorial are as follows:

* [Establishing Connectivity - the Engine](https://docs.sqlalchemy.org/en/20/tutorial/engine.html#tutorial-engine) - all SQLAlchemy applications start with an **Engine** object; here's how to create one.
* [Working with Transactions and the DBAPI](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-working-with-transactions) - the usage API of the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) and its related objects [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) are presented here. This content is Core-centric however ORM users will want to be familiar with at least the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) obejct.
* [Working with Database Metadata](https://docs.sqlalchemy.org/en/20/tutorial/metadata.html#tutorial-working-with-metadata) - SQLAlchemy's SQL abstractions as well as the ORM rely upon a system of defining database schema consructs as Python objects. This section introduces how to do that from both a Core and an ORM perspective.
* [Working with Data](https://docs.sqlalchemy.org/en/20/tutorial/data.html#tutorial-working-with-data) - here we learn how to create, select, update and delete data in the database. The so-called [CRUD](https://docs.sqlalchemy.org/en/20/glossary.html#term-CRUD) operations here are given in terms of SQLAlchemy Core with links out towards their ORM counterparts. The SELECT operation that is introduced in detail at [Using SELECT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html#tutorial-selecting-data) applies equally well to Core and ORM.
* [Data Manipulation with the ORM](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-data-manipulation) covers the persistence framework of the ORM; basically the ORM-centric ways to insert, update and delete, as well as how to handle transactions.
* [Worknig with ORM Related Objects](https://docs.sqlalchemy.org/en/20/tutorial/orm_related_objects.html#tutorial-orm-related-objects) introduces the concept of the [relationship()](https://docs.sqlalchemy.org/en/20/orm/relationship_api.html#sqlalchemy.orm.relationship) construct and provides a brief overview of how it's used, with links to deeper documentation.
* [Further Reading](https://docs.sqlalchemy.org/en/20/tutorial/further_reading.html#tutorial-further-reading) lists a series of major top-level documentation sections which fully document the concepts introduced in this tutorial.

In [1]:
import sqlalchemy
print(sqlalchemy.__version__)

2.0.45


# Establishing Connectivity - the Engine

## Welcome ORM and Core readers alike!

Every SQLAlchemy application that connects to a database needs to use an [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine). This short section is for everyone.

The start of any SQLAlchemy application is an object called the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine). This object acts as a central source of connections to a particular database, providing both a factor as well as a holding space called a [connection pool](https://docs.sqlalchemy.org/en/20/core/pooling.html) for these database connections. The engine is typically a global object created just once for a particular database server, and is configured using a URL string which will describe how it should connect to the datbase host or backend.

For this tutorial we will use SQLite database. The [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is created by using the [create_engine()](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine) function:

In [2]:
from sqlalchemy import create_engine
dbPath = 'tutorial.db'
engine = create_engine('sqlite+pysqlite:///%s' % dbPath, echo=True)

To use an in-memory-only SQLite database, which is an easy way to test things without needing to have an actual pre-existing database set up:

```python
from sqlalchemy import create_engine
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)
```

The main argument to [create_engine](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine) is a string URL, above passed as the string `"sqlite_pysqlite:///tutorial.db"`. This string indicates to the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) three important facts:

1. What kind of database are we communicating with? This is the `sqlite` portion above, which links in SQLAlchemy to an object known as the [dialect](https://docs.sqlalchemy.org/en/20/glossary.html#term-dialect).
2. What [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) are we using? The Python [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) is a third party driver that SQLAlchemy uses to interact with a particular database. In this case, we're using the name `pysqlite`, which in modern Python use is the [sqlite3](https://docs.python.org/library/sqlite3.html) standard library interface for SQLite. It omitted, SQLAlchemy will use a default [DBAPI](https://docs.sqlalchemy.org/en/20/glossary.html#term-DBAPI) for the particular database selected.
3. How do we locate the database? In this case, our URL could include the phrase `/:memory:`, which is an indicator to the `sqlite3` module that we will be using an **in-memory-only** database. This kind of database is perfect for experimenting as it does not require any server nor does it need to create new files.

In [3]:
'sqlite+pysqlite:///%s' % dbPath

'sqlite+pysqlite:///tutorial.db'

## Lazy Connecting

The [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine), when first returned by [create_engine()](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine), has not actually tried to connect to the database yet; that happens only the first time it is asked to perform a task against the database. This is a software design pattern known as [lazy initialization](https://docs.sqlalchemy.org/en/20/glossary.html#term-lazy-initialization).

We have also specified a parameter [create_engine.echo](https://docs.sqlalchemy.org/en/20/core/engines.html#sqlalchemy.create_engine.params.echo), which will instruct the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) to log all of the SQL it emits to a Python logger that will write to standard out. This flag is a shorthand way of setting up [Python logging more formally](https://docs.sqlalchemy.org/en/20/core/engines.html#dbengine-logging) and is useful for experimentation in scripts. Many of the SQL examples will include this SQL logging output beneath a [ SQL ] link that when clicked, will reveal the full SQL interaction.

# Working with Transactions and the DBAPI

With the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) object ready to go, we can dive into the basic operation of an [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) and its primary endpoints, the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result). We'll also introduce the ORM's [facade](https://docs.sqlalchemy.org/en/20/glossary.html#term-facade) for these objects, known as the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session)


## Note to ORM readers

When using the ORM, the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is managed by the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session). The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) in modern SQALchemy emphasizes a transactional and SQL execution pattern that is largely identical to that of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) discussed below, so while this subsection is Core-centric, all of the concepts here are relevant to ORM use as well and is recommended for all ORM learners. The execution pattern used by the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) will be compared to the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) at the end of this section.


As we have yet to introduce the SQLAlchemy Expression Language that is the primary feature of SQLAlchemy, we'll use a simple construct within this package called the [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) construct to write SQL statements, as **textual SQL**. Rest assured that textual SQL is the exception rather than the rule in day-to-day SQLAlchemy use, but it's always available.


## Getting a Connection

The purpose of the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) is to connect to the database by providing a [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection). When working with the Core directly, the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object is how all interaction with the database is done. Because the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) creates an open resource against the database, we want to limit our use of this object to a specific context. The best way to do that is with a Python context manager, also known as [the with statement](https://docs.python.org/3/reference/compound_stmts.html#with). Below we use a textual SQL statement to show "Hello World". Textual SQL is created with a construct called [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) which we'll discuss in more detail later.

In [4]:
from sqlalchemy import text
with engine.connect() as conn:
    result = conn.execute(text("SELECT 'hello world'"))
    print(result.all())

2026-01-23 08:52:19,190 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,190 INFO sqlalchemy.engine.Engine SELECT 'hello world'
2026-01-23 08:52:19,191 INFO sqlalchemy.engine.Engine [generated in 0.00130s] ()
[('hello world',)]
2026-01-23 08:52:19,192 INFO sqlalchemy.engine.Engine ROLLBACK


In the example above, the context manager creates a database connectoin and executes the operation in a transaction. the default behavior of teh Python DBAPI is that a transaction is always in progress; when the connection is [released](https://docs.sqlalchemy.org/en/20/glossary.html#term-released), a ROLLBACK is emitted to end the transactoin. The transaction is **not committed automatically**; if we want to commit data we need to call [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) as we'll see in the next section.


### Tip

"autocommit" mode is available for special cases. The section [Setting Transaction Isolation Levels Including DBAPI Autocommit](https://docs.sqlalchemy.org/en/20/core/connections.html#dbapi-autocommit) discusses this.


The result of our SELECT was returned in an object called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) that will be discussed later. For the moment we'll add that it's best to use this object within the "connect" block, and not ot use it outside of the scope of our connection.


# Committing Changes

We just learned that the DBAPI connection doesn't commit automatically. What if we want to commit some data? We can change our example above to create a table, insert some data and then commit the transaction using the [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) method, **inside** the block where we have the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object:

In [5]:
# "commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-23 08:52:19,203 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,205 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-23 08:52:19,205 INFO sqlalchemy.engine.Engine [generated in 0.00231s] ()
2026-01-23 08:52:19,225 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 08:52:19,225 INFO sqlalchemy.engine.Engine [generated in 0.00085s] [(1, 1), (2, 4)]
2026-01-23 08:52:19,227 INFO sqlalchemy.engine.Engine COMMIT


Above, we execute two SQL statements, a "CREATE TABLE" statement and an "INSERT" statement that's parametrized (we discuss the parametrization syntax later in [Sending Multiple Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-multiple-parameters)). To commit the work we've done in our block, we call the [Connectoin.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) method which commits the transaction. After this, we can continue to run more SQL statements and call [Connection.commit()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.commit) again for those statements. SQLAlchemy refers to this style as **commit as you go**.


There's also another style to commit data. We can declare our "connect" block to be a transactoin block up front. to do this, we use the [Engine.begin()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine.begin) method to get the connection, rather than the [Engine.connect()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine.connect) method. This method will manage the scope of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) and also enclose everything inside of a transaction with either a COMMIT at the end if the block was successful, or a ROLLBACK if an exception was raised. This style is known as **begin once**:

In [6]:
# "begin once"
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-23 08:52:19,247 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,249 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 08:52:19,250 INFO sqlalchemy.engine.Engine [cached since 0.02554s ago] [(6, 8), (9, 10)]
2026-01-23 08:52:19,252 INFO sqlalchemy.engine.Engine COMMIT


You should mostly prefer the "begin once" style because it's shorter and shows the intention of the entire block up front. However, in this tutorial we'll use the "commit as you go" style as its more flexible for demonstration purposes.

## What's "BEGIN (implicit)"?

You might have noticed the log line "BEGIN (implicit)" at the start of a transaction block. "implicit" here means that SQLAlchemy **did not actually send any command** to the database; it just considers this to be the start of the DBAPI's implicit transaction. You can register [event hooks](https://docs.sqlalchemy.org/en/20/core/events.html#core-sql-events) to intercept this event, for example.


[DDL](https://docs.sqlalchemy.org/en/20/glossary.html#term-DDL) refers to the subset of SQL that instructs the database to create, modify, or remove schema-level constructs such as tables. DDL such as "CREATE TABLE" should be in a transaction block that ends with COMMIT, as many databases uses transactional DDL such that the schema changes don't take place until the transaction is committed. However, as we'll see later, we usually let SQLAlchemy run DDL sequences for us as part of a higher level operatoin where we don't generally need to worry about COMMIT.



## Basics of Statement Execution

We have seen a few examples that run SQL statements against a database, making use of a method called [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute), in conjunction with an object called [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text), and returning an object called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result). In this section we'll illustrate more closely the mechanics and interactions of these components.


Most of the content of this section applies equally well to modern ORM use when using the [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method, which works very similarly to that of [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute), including that ORM result rows are delivered using the same [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) interface used by Core.


## Fetching Rows

We'll first illustrate the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) object more closely by making use of the rows we've inserted previously, running a textual SELECT statement on the table we've created:

In [7]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y from some_table"))
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-23 08:52:19,280 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,282 INFO sqlalchemy.engine.Engine SELECT x, y from some_table
2026-01-23 08:52:19,282 INFO sqlalchemy.engine.Engine [generated in 0.00200s] ()
x: 1, y: 1
x: 2, y: 4
x: 6, y: 8
x: 9, y: 10
2026-01-23 08:52:19,284 INFO sqlalchemy.engine.Engine ROLLBACK


Above, the "SELECT" string we executed selected all ows from our table. The object returned is called [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) and represents an iterable object of the result rows.

[Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) has lots of methods for fetching and transforming rows, such as the [Result.all()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result.all) method illustrated previously, which returns a list of all [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects. It also implements the Python iterator interface so that we can iterate over the collection of [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects directly.

The [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects themselves are intended to act like Python [named tuples](https://docs.python.org/3/library/collections.html#collections.namedtuple)

Below we illustrate a variety of ways to acccess rows

* **Tuple Assignment** - This is the most Python-idiomataic style, which is to assign variables to each row positionally as they are are received:

```python
result = conn.execute(text("select x, y from some_table"))

for x, y in result:
    ...
```


* **Integer Index** - Tuples are Python sequences, so regular integer access is available too:

```python
result = conn.execute(text("select x, y from some_table"))

for row in result:
    x = row[0]
```


* **Attribute Name** - As these are Python named tuples, the tuples have dynamic attribute names matching the names of each column. These names are normally the names that the SQL statement assigns to the columns in each row. While they are usually fairly predictable and can also be controlled by labels, in less defined cases they may be subject to database-specific behaviors:

```python
result = conn.execute(text("select x, y from some_table"))

for row in result:
    y = row.y

    # illustate use with Python f-strings
    print(f"Row: {row.x} {y}")
```


* **Mapping Access** - To receive rows as Python **mapping** objects, which is essentially a read-only version of Python's interface to the common `dict` object, the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) may be **transformed** into a [MappingResult](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.MappingResult) object using the [Result.mappings()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result.mappings) modifier; this is a result object that yields dictionary-like [RowMapping](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.RowMapping) objects rather than [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects:

```python
result = conn.execute(text("select x, y from some_table"))

for dict_row in result.mappings():
    x = dict_row["x"]
    y = dict_row["y"]
```

## Sending Parameters

SQL statements are usually accompanied by data that is to be passed with the statement itself, as we saw in the INSERT example previously. The [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method therefore also accepts parameters, which are known as [bound parameters](https://docs.sqlalchemy.org/en/20/glossary.html#term-bound-parameters). A rudimentary example might be if we wanted to limit our SELECT statement only to rows that meet a certain criteria, such as rows where the "y" value are greater than a certain value that is passed in to a function.

In order to achieve this such that the SQL statement can remain fixed and that the driver can properly sanitize the vlaues, we add a WHERE criteria to our statement that names a new parameter called "y", the [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) construct accepts these using a colon format ":y". The actual value for ":y" is then passed as the second argument to [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) in the form of a dictionary:

In [8]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 2})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-23 08:52:19,298 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,299 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-23 08:52:19,300 INFO sqlalchemy.engine.Engine [generated in 0.00252s] (2,)
x: 2, y: 4
x: 6, y: 8
x: 9, y: 10
2026-01-23 08:52:19,302 INFO sqlalchemy.engine.Engine ROLLBACK


In the logged SQL output, we can see that the bound parameter :y was converted into a question mark when it was sent to the SQLite database. This is because the SQLite database driver uses a format called "qmark parameter style", which is one of six different formats allowed by the DBAPI specification. SQLAlchemy abstracts these formats into just one, which is the "named" format with a colon.


### Always use bound parameters

As mentioned at the beginning of this section, textual SQL is not the usual way we work with SQLAlchemy. However, when using textual SQL, a Python literal value, even non-strings like integers or dates, should **never be stringified into SQL string directly**: a parameter should **always** be used. This ismost famously known as how to avoid SQL injection attacks when the data is untrusted. However, it also allows the SQLAlchemy dialects and/or DBAPI to correctly handle the incoming input for the backend. Outside of plain textual SQL use cases, SQlAlchemy's Core Expression API otherwise ensures that Python literal values are passed as bound parameters where appropriate. 

## Sending Multiple Parameters

In the example at [Committing Changes](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-committing-data), we executed an INSERT statement where it appeared that we were able to INSERT multiple rows into the database at once. For [DML](https://docs.sqlalchemy.org/en/20/glossary.html#term-DML) statements such as "INSERT", "UPDATE", and "DELETE", we can send **multiple parameter sets** to the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method by passing a list of dictionaries instead of a single dictionary, which indicates that the single SQL statement should be invoked multiple times, once for each parameter set. This style of execution is known as [executemany](https://docs.sqlalchemy.org/en/20/glossary.html#term-executemany):

In [9]:
with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}],
    )
    conn.commit()

2026-01-23 08:52:19,315 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,316 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 08:52:19,317 INFO sqlalchemy.engine.Engine [cached since 0.09257s ago] [(11, 12), (13, 14)]
2026-01-23 08:52:19,319 INFO sqlalchemy.engine.Engine COMMIT


The above operation is equivalent to running the given INSERT statement once for eahc parameter set, except that the operation will be optimized for better performance across many rows.

A key behavioral difference between "execute" and "executemany" is that the latter doesn't support returning results of rows, even if the statement includes the RETURNING clause. The one exception to this is when using a Core [insert()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.insert) construct, introduced later in this tutorial at [Using INSERT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_insert.html#tutorial-core-insert), which also indicates RETURNING using the [Insert.returning()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.returning) method. In that case, SQLAlchemy makes use of special logic to reorganize the INSERT statement so that it can be invoked for many rows while stil support RETURNING.

### See Also

[executemany](https://docs.sqlalchemy.org/en/20/glossary.html#term-executemany) in the [Glossary](https://docs.sqlalchemy.org/en/20/glossary.html) describes the DBAPI-level [cursor.executemany()](https://peps.python.org/pep-0249/#executemany) method that's used for most "executemany" executions.  ["Insert Many Values" Behavior for INSERT statements](https://docs.sqlalchemy.org/en/20/core/connections.html#engine-insertmanyvalues) - in [Working with Engines and Connections](https://docs.sqlalchemy.org/en/20/core/connections.html), describes the specialized logic used by [Insert.returning()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.returning) to deliver result sets with "executemany" executions.

## Executing with an ORM Session

As mentioned previously, most of the patterns and examples above apply to use with the ORM as well, so here we will introduce this usage so that as the tutorial proceeds, we will be able to illustrate each pattern in terms of Core and ORM uses together.

The fundamental transactional / database interactive object when using the ORM is called the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session). In modern SQLAlchemy, this object is used in a manner very similar to that of the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection), and in fact as the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) is used, it refers to a [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) internally, which is uses to emit SQL.


When the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) is used with non-ORM constucts, it passes through the SQL statements we give it and does not generally do things much differently from how the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) does directly, so we can illustrate it here in terms of the simple textual SQL operations we've already learned.

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) has a few different creational patterns, but here we will illustrate the most basic one that tracks exactly with how the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) is used which is to construct it within a context manager.

https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html

In [10]:
from sqlalchemy.orm import Session

stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-23 08:52:19,422 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,423 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-23 08:52:19,424 INFO sqlalchemy.engine.Engine [generated in 0.00064s] (6,)
x: 6 y: 8
x: 9 y: 10
x: 11 y: 12
x: 13 y: 14
2026-01-23 08:52:19,426 INFO sqlalchemy.engine.Engine ROLLBACK


The example above can be compared to the example in the preceding section in [Sending Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-sending-parameters) - we diretly replace the call to `with engine.connect() as conn` with `with Session(engine) as session`, and then make use of the [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) just like we do with the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute)

Also, like the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection), the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) features "commit as you go" behavior using the [Session.commit()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.commit) method, illustrated below using a textual UPDATE statement to alter some of our data:

In [11]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}],
    )
    session.commit()

2026-01-23 08:52:19,435 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,436 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-23 08:52:19,437 INFO sqlalchemy.engine.Engine [generated in 0.00129s] [(11, 9), (15, 13)]
2026-01-23 08:52:19,439 INFO sqlalchemy.engine.Engine COMMIT


Above, we invoked an UPDATE statement using the bound-parameter, "executemany" style of execution introducted at [Sending Multiple Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-multiple-parameters), ending the block with a "commit as you go" commit.


### Tip

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) doesn't actually hold onto the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) object after it ends the transaction. It gets a new [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) from the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) the next time it needs to execute SQL against the database.

The [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) obviously has a lot more tricks up its sleeve than that, however understanding that it has a [Session.execute](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method that's used the same way as [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) will get us started with the examples that follow later.

# Working with Database Metadata

With engines and SQL execution down, we are ready to begin some Alchemy. The central element of both SQLAlchemy Core and ORM is the SQL Expression Language which allows for fluent, composable construction of SQL queries. The foundation for these queries are Python objects that represent database concepts like tables and columns. these objects are known collectively as [database metadata](https://docs.sqlalchemy.org/en/20/glossary.html#term-database-metadata). 

The most common foundatoinal objects for database metadta in SQLAlchemy are known as [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData), [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table), and [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column). The sections below will illustrate how these objects are used in both a Core-oriented style as well as an ORM-oriented style.


### ORM readers, stay with us!

As with other sections, Core users can skip the ORM sections, but ORM users would best be familiar with these objects from both perspectives. The [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) discussed here is declared in a more indirect (and also fully Python-typed) way when using the PRM, however there is still a [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) wtihin the ORM's configuration.


# Setting up MetaData with Table objects

When we work with a relational database, the basic data-holding structure in the database which we query from is known as a **table**. in SQLAlchemy, the database "table" is ultimately represented by a Python object similarly named [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table).


To start using the SQLAlchemy Expression Language, we will want to have [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects constructed that represent all of the database tables we are interested in working with. The [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) is constructed programmatically, either by using the [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) constructor, or indirectly by using ORM Mapped classes (described later at [Using ORM Declarative Forms to Define Table Metadata](https://docs.sqlalchemy.org/en/20/tutorial/metadata.html#tutorial-orm-table-metadata)). There is also the option to load some or all table information from an existing database, called [reflection](https://docs.sqlalchemy.org/en/20/glossary.html#term-reflection).


Whichever kind of approach is used, we always start out with a collection that will be where we place our tables known as the [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) object. This object is essentially a [facade](https://docs.sqlalchemy.org/en/20/glossary.html#term-facade) around a Python dictionary that stores a series of [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects keyed to their string name. While the ORM provides some options on where to get this collection, we always have hte optino to simply make one directly, which looks like:

In [12]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

Once we have a [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) object, we can declare some [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects. This tutorial will start with the classic SQLAlchemy tutorial model, which has a table called `user_account` that stores, for example, the users of a website, and a related table `address`, which stores email addresses associated with rows in the `user_account` table. When not using ORM declarative models at all, we construct each [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) object directly, typically assigning each to a variable that will be how we will refer to the table in application code:

In [13]:
from sqlalchemy import Table, Column, Integer, String, Float
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

With the above example, when we wish to write code that refers to the `user_account` table in the database, we will use the `user_table` Python variable to refer to it.

### When do i make a `MetaData` object in my program?


Having a single [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) object for an entire application is the most common case, represented as a module-level variable in a single place in an application, often in a "models" or "dbschema" type of package. It is also very common that the [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) is accessed via an ORM-centric [registry](https://docs.sqlalchemy.org/en/20/orm/mapping_api.html#sqlalchemy.orm.registry) od [Declarative Base](https://docs.sqlalchemy.org/en/20/tutorial/metadata.html#tutorial-orm-declarative-base) base class, so that this same [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) is shared among ORM- and Core-declared [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects.


There can be multiple [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) collections as well; [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects can refer to [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects in other collections without restrictions. However, for groups of [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects that are related to each other, it is in practice much more straightforward to have them set up within a single [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) collection, both from the perspective of declaring them, as well as from the perspective of DDL (i.e. CREATE and DROP) statements being emitted in the correct order.

## Components of `Table`

We can observe that the [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) construct as written in Python has a resemblance to a SQL CREATE TABLE statement, starting with the table name, then listing out each column, where each column has a name and a datatype. The objects we use above are:

* [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) - represents a database table and assigns itself to a [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) collection.
* [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) - represents a column in a database table, and assigns itself to a [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) object. The [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) usually includes a string name and a type object. The collection of [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) objects in terms of the parent [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) are typically accessed via an associative array located at [Table.c](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table.c):

In [14]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [15]:
user_table.c.keys()

['id', 'name', 'fullname']

* [Integer](https://docs.sqlalchemy.org/en/20/core/type_basics.html#sqlalchemy.types.Integer), [String](https://docs.sqlalchemy.org/en/20/core/type_basics.html#sqlalchemy.types.String) - these classes represent SQL datatypes and can be passed to a [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) with or without necessarily being instantiated. Above, we want to give a length of "30" to the "name" column, so we instantiated `String(30)`. But for "id" and "fullname" we did not specify these, so we can send the class itself.


### See also


The reference and API documentation for [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData), [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) and [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) is at [Describing Databases with MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html). The reference documentation for datatypes is at [SQL Datatype Objects](https://docs.sqlalchemy.org/en/20/core/types.html). 


In an upcoming section, we will illustrate one of the fundamental functions of [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) which is to generate [DDL](https://docs.sqlalchemy.org/en/20/glossary.html#term-DDL) on a particular database connection. But first we will declare a second [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table).

## Declaring Simple Constraints

The first [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) in the example `user_table` includes the [Column.primary_key](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column.params.primary_key) parameter which is a shorthand technique of indicating that this [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) should be part of the primary key for this table. The primary key itself is noramlly declared implicitly and is represented by the [PrimaryKeyConstraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.PrimaryKeyConstraint) construct, which we can see on the [Table.primary_key](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table.primary_key) attribute on the [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) object:

In [16]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

The constraint that is most typically declared explicitly is the [ForeignKeyConstraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.ForeignKeyConstraint) object that corresponds to a database [foreign key constraint](https://docs.sqlalchemy.org/en/20/glossary.html#term-foreign-key-constraint). When we declare tables that are related to each other, SQLAlchemy uses the presence of these foreign key constraint declarations not only so that they are emitted within CREATE statements to the database, but also to asist in constructing SQL expressions.

A [ForeignKeyConstraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.ForeignKeyConstraint) that involves only a single column on the target table is typically declared using a column-level shorthand notatoin via the [ForeignKey](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.ForeignKey) object. Below we declare a second table `address` that will have a foreign key constraint referring to the `user` table:

In [17]:
from sqlalchemy import ForeignKey
address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False),
)

The table above features a third kind of constraint, which in SQL is the "NOT NULL" constraint, indicated above using the [Column.nullable](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column.params.nullable) parameter.


### Tip

When using the [ForeignKey](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.ForeignKey) object within a [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) definition, we can omit the datatype for that [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column); it is automatically inferred from that of the related column, in the above example the [Integer](https://docs.sqlalchemy.org/en/20/core/type_basics.html#sqlalchemy.types.Integer) datatype of the `user_account.id` column.

In the next section we will emit the completed DDL for the `user` and `address` table to see the completed result.

## Emitting DDL to the Database

We've constructed an object structure that represents two database tables in a database, starting at the root [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) object, then into two [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects, each of which hold onto a collection of [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) and [Constraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#sqlalchemy.schema.Constraint) objects. This object structure will be at the center of most operations we perform with both Core and ORM going forward. 

The first useful thing we can do with this structure will be to emit CREATE TABLE statements, or [DDL](https://docs.sqlalchemy.org/en/20/glossary.html#term-DDL), to our SQLite database so that we can insert and query data from them. We have already all the tools needed to do so, by invoking the [MetaData.create_all()](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData.create_all) method on our [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData), sending it to the [Engine](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Engine) that refers to the target database:

In [18]:
metadata_obj.create_all(engine)

2026-01-23 08:52:19,533 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,534 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-23 08:52:19,535 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,536 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-01-23 08:52:19,537 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,537 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-23 08:52:19,538 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,539 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-01-23 08:52:19,539 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,541 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-01-23 08:52:19,541 INFO sqlalchemy.engine.Engine [no key 0.00055s] ()
2026-01-23 08:52:19,564 INFO sqlalchemy.engine.Engine 
C

The DDL create process above includes some SQLite-specific PRAGMA statements that test for the existence of each table before emitting a CREATE. The full series of steps are also included within a BEGIN/COMMIT pair to accomodate for transactional DDL.


The create process also takes care of emitting CREATE statements in the correct order; above, the FOREIGN KEY constraint is dependent on the `user` table existing, so the `address` table is created second. In more complicated dependency scenarios the FOREIGN KEY constraints may also be applied to tables after the fact using ALTER.


The [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) object also features a [MetaData.drop_all()](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData.drop_all) method that will emit DROP statements in the reverse order as it would emit CREATE in order to drop schema elements.

## Migration tools are usually appropriate

Overall, the CREATE / DROP feature of [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) is useful for test suites, small and/or new applications, and applications that use short-lived databases. For management of an application database schema over the long term however, a schema management tool such as [Alembic](https://alembic.sqlalchemy.org/), which builds upon SQLAlchemy, is likely a better choice, as it can manage and orchestrate the process of incrementally altering a fixed database over time as the design of the application changes.


## Using ORM Declarative Forms to Define Table Metadata

### Another way to make Table objects?

The preceding examples illustrated direct use of the [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) object, which underlies how SQLAlchemy ultimately refers to databse tables when constructing SQL expressions. As mentioned, the SQLAlchemy ORM provides for a facade around the [Talbe](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) declaration process towards as **Declarative Table**. The Declarative Table process accomplishes the same goal as we had in the prevoius section, that of building [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) objects, but also within that process gives us something else called an [ORM mapped class](https://docs.sqlalchemy.org/en/20/glossary.html#term-ORM-mapped-class), or just "mapped class". The mapped class is the most common foundatoinal unit of SQL when using the ORM, and in modern SQLAlchemy can also be used quite effectively with Core-centric use as well.

Some benefits of using Declarative Table include:

* A more succint and Pythonic style of setting up column definitions, where Python types may be used to represent SQL types to be used in the database
* The resulting mapped class can be used to form SQL expressions that in many cases maintain [PEP 484](https://peps.python.org/pep-0484/) typing information that's picked up by static analysis tools such as Mypy and IDE type checkers
* Allows declaration of table metadata and the ORM mapped class used in persistence / object loading operations all at once.

This section will illustrate the same [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) metadata of the previous section(s) being constructed using Declarative Table.


When using the ORM, the process by which we declare [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) metadata is usually combined with the process of declaring [mapped](https://docs.sqlalchemy.org/en/20/glossary.html#term-mapped) classes. The mapped class is any Python class we'd like to create, which will then have attributes on it that will be linked to the columns in a database table. While there are a few varieties of how this is achieved, the most common style is known as [declarative](https://docs.sqlalchemy.org/en/20/orm/declarative_config.html), and allows us to declare our user-defined classes and [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) metadata at once.

In [19]:
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [20]:
Base.metadata

MetaData()

In [21]:
Base.registry

In [22]:
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"


class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))
    email_address: Mapped[str]

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, user_id={self.user_id!r}, email_address={self.email_address!r})"

In [23]:
sandy = User(name="Sandy", fullname="Sandy Cheeks")
sandy

User(id=None, name='Sandy', fullname='Sandy Cheeks')

In [24]:
Base.metadata.create_all(engine)

2026-01-23 08:52:19,665 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,666 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-23 08:52:19,667 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,670 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-23 08:52:19,670 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,671 INFO sqlalchemy.engine.Engine COMMIT


In [25]:
some_table = Table("some_table", metadata_obj, autoload_with=engine)

2026-01-23 08:52:19,682 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,688 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("some_table")
2026-01-23 08:52:19,689 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,691 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2026-01-23 08:52:19,692 INFO sqlalchemy.engine.Engine [raw sql] ('some_table',)
2026-01-23 08:52:19,694 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("some_table")
2026-01-23 08:52:19,695 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,696 INFO sqlalchemy.engine.Engine PRAGMA temp.foreign_key_list("some_table")
2026-01-23 08:52:19,697 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-23 08:52:19,698 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type i

In [26]:
some_table

Table('some_table', MetaData(), Column('x', INTEGER(), table=<some_table>), Column('y', INTEGER(), table=<some_table>), schema=None)

https://docs.sqlalchemy.org/en/20/tutorial/data.html

# Working with Data

in [Working with Transactions and the DBAPI](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-working-with-transactions), we learned the basics of how to interact with the Python DBAPI and its transactional state. Then, in [Wroking with Database Metadata](https://docs.sqlalchemy.org/en/20/tutorial/metadata.html#tutorial-working-with-metadata), we learned how to represent database tables, columns, and constraints wtihin SQLAlchemy using the [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData) and related objects. In this section we will combine both concepts above to create, select and manipulate data within a relational database. Our interaction with the datbase is **always** in terms of a transaction, even if we've set our datbase driver to use [autocommit](https://docs.sqlalchemy.org/en/20/core/connections.html#dbapi-autocommit) behind the scenes.

The components of this section are as follows:

* [Using INSERT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_insert.html#tutorial-core-insert) - to get some data into the database, we introduce and demonstrate the Core [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct. INSERTs from an ORM perspective are described in the next section [Data Manipulation with the ORM](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-data-manipulation)
* [Using SELECT Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html#tutorial-selecting-data) - this section will describe in detail the [Select](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.Select) construct, which is the most commonly used object in SQLAlchemy. The [Select](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.Select) construct emits SELECT statements for both Core and ORM centric applications and both use cases will be described here. Additional ORM use cases are also noted in the later section [Using Relationships in Queries](https://docs.sqlalchemy.org/en/20/tutorial/orm_related_objects.html#tutorial-select-relationships) as well as the [ORM Querying Guide](https://docs.sqlalchemy.org/en/20/orm/queryguide/index.html).
* [using UPDATE and DELETE Statements](https://docs.sqlalchemy.org/en/20/tutorial/data_update.html#tutorial-core-update-delete) - Rounding out the INSERT and SELECTion of data, this section will describe from a Core perspective the use of the [Update](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Update) and [Delete](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Delete) constructs. ORM-specific UPDATE and DELETE is similarly described in the [Data Manipulation with the ORM](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-data-manipulation) section.

# Using INSERT Statements

When using Core as well as when using the ORM for bulk operations, a SQL INSERT statement is generated directly using the [insert()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.insert) function - this function generates a new instance of [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) which represents an INSERT statement in SQL, that adds new data into a table.

### ORM Readers

This section details the Core means of generating an individual SQL INSERT statement in order to add new rows to a table. When using the ORM, we normally use another tool that rides on top of this called the [unit of work](https://docs.sqlalchemy.org/en/20/glossary.html#term-unit-of-work), which will automate the production of many INSERT statements at once. However, understanding how the core handles data creation and manipulation is very useful even when the ORM is running it for us. Additionally, the ORM supports direct use of INSERT using a feature called [Bulk / Multi Row INSERT, upsert, UPDATE and DELETE](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-bulk)

To skip directly to how to INSERT rows with the ORM using normal unit of work patterns, see [Inserting Rows using the ORM unit of Work pattern](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-inserting-orm)


# The insert() SQL Expression Construct

A simple example of [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) illustrating the target table and the VALUES clause at once:

In [27]:
some_table

Table('some_table', MetaData(), Column('x', INTEGER(), table=<some_table>), Column('y', INTEGER(), table=<some_table>), schema=None)

In [28]:
user_table

Table('user_account', MetaData(), Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False), Column('name', String(length=30), table=<user_account>), Column('fullname', String(), table=<user_account>), schema=None)

In [29]:
from sqlalchemy import insert
stmt = insert(user_table).values(name="spongebob",fullname="Spongebob Squarepants")

The above `stmt` variable is an instance of [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert). Most SQL expressions can be stringified in place as a means to see the general form of what's being produced: 

In [30]:
print(stmt)

INSERT INTO user_account (name, fullname) VALUES (:name, :fullname)


The stringified form is created by producing a [Compiled](https://docs.sqlalchemy.org/en/20/core/internals.html#sqlalchemy.engine.Compiled) form of the object which includes a database-specific string SQL representation of the statement; we can acquire this object dirctly using the [ClauseElement.compile()](https://docs.sqlalchemy.org/en/20/core/foundation.html#sqlalchemy.sql.expression.ClauseElement.compile) method.

In [31]:
compiled = stmt.compile()

Our [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct is an example of a "parametrized" construct, illustrated previously at [Sending Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-sending-parameters); to view the `name` and `fullname` [bound parameters](https://docs.sqlalchemy.org/en/20/glossary.html#term-bound-parameters), these are available from the [Compiled](https://docs.sqlalchemy.org/en/20/core/internals.html#sqlalchemy.engine.Compiled) construct as well:

In [32]:
compiled.params

{'name': 'spongebob', 'fullname': 'Spongebob Squarepants'}

# Executing the Statement

Invoking the statement we can INSERT a row into `user_table`. The INSERT SQL as well as the bundled parameters can be seen in the SQL logging:

In [33]:
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

2026-01-23 08:52:19,810 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,810 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-01-23 08:52:19,811 INFO sqlalchemy.engine.Engine [generated in 0.00153s] ('spongebob', 'Spongebob Squarepants')
2026-01-23 08:52:19,812 INFO sqlalchemy.engine.Engine COMMIT


In its simple form above, the INSERT statement does not return any rows, and if only a single row is inserted, it will usually include the ability to return information about column-level default values that were generated during the INSERT of that row, most commonly an integer primary key value. In the above case the first row in a SQLite database will normally return `1` for the first integer primary key value, which we can acquire using the [CursorResult.inserted_primary_key](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key) accessor:

In [34]:
result.inserted_primary_key

(1,)

### Tip

[CurosorResult.inserted_primary_key](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key) returns a tuple because a primary key may contain multiple columns. This is known as a [composite primary key](https://docs.sqlalchemy.org/en/20/glossary.html#term-composite-primary-key). The [CursorResult.inserted_primary_key](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key) is intended to always contain the complete primary key of the record just inserted, not just a "cursor.lastrowid" kind of value, and is also intended to be populated regardless of whether or not "autoincrement" were used, hence to express a complete primary key it's a tuple.

In [35]:
type(result.inserted_primary_key) # a named tuple actually

sqlalchemy.engine.row.Row

In [36]:
pk = result.inserted_primary_key
pk

(1,)

the tuple returned by [CursorResult.inserted_primary_key](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.CursorResult.inserted_primary_key) is now a named tuple fulfilled by returning it as a [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) object.

In [37]:
pk.id

1

# INSERT usually generates the "values" clause automatically

The example above made use of the [insert.values()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.values) method to explicitly create the VALUES clause of the SQL INSERT statement. If we don't actually use [Insert.values()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.values) and just print out an "empty" statement, we get an INSERT for every column in the table:

In [38]:
print(insert(user_table))

INSERT INTO user_account (id, name, fullname) VALUES (:id, :name, :fullname)


If we take an [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct that has not had [Insert.values()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.values) called upon it and execute it rather than print it, the statement will be compiled to a string based on the parameters that we passed to the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method, and only include columns relevant to the parameters that were passed. This is actually the usual way that [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) is used to insert rows without having to type out an explicit VALUES clause. The example below illustrates a two-column INSERT statemetn being executed with a list of parameters at once.

In [39]:
with engine.connect() as conn:
    result = conn.execute(
        insert(user_table),
        [
            {"name": "sandy", "fullname": "Sandy Cheeks"},
            {"name": "patrick", "fullname": "Patrick Star"},
        ]
    )
    conn.commit()

2026-01-23 08:52:19,897 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,897 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-01-23 08:52:19,898 INFO sqlalchemy.engine.Engine [generated in 0.00115s] [('sandy', 'Sandy Cheeks'), ('patrick', 'Patrick Star')]
2026-01-23 08:52:19,899 INFO sqlalchemy.engine.Engine COMMIT


The execution above features "executemany" form first illustrated at [Sending Multiple Parameters](https://docs.sqlalchemy.org/en/20/tutorial/dbapi_transactions.html#tutorial-multiple-parameters), however unlike when using the [text()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.text) construct, we didn't have to spell out any SQL. By passing a dictionary or list of dictionaries to the [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) method in conjunction with the [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct, the [Connection](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection) ensures that the column names which are passed will be expressed in the VALUES clause of the [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct automatically.


### Tip

When passing a list of dictionaries to [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) along with a Core [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert), **only the first dictionary in the list determines what columns will be in the VALUES clause**. The rest of the dictionaries are not scanned. This is both because within traditional `executemany()`, the INSERT statement can only have one VALUES clause for all parameters, and additionally SQLAlchemy does not want to add overhead by scanning every parameter dictionary to verify each contains idential keys as the first one.

Note this behavior is distinctly different from that of an [ORM enabled INSERT](https://docs.sqlalchemy.org/en/20/tutorial/orm_data_manipulation.html#tutorial-orm-bulk), introduced later in this tutorial, which performs a full scan of parameter sets in terms of an ORM entity.

## Deep Alchemy

Hi, welcome to the first edition of **Deep Alchemy**. The person on the left is known as **The Alchemist**, and you'll note that they are **not** a wizard, as the pointy hat is not sticking upwards. The Alchemist comes around to describe things that are generally **more advanced and/or tricky** and additionally **not usually needed**, but for whatever reason they feel you should know about this thing that SQLAlchemy can do.

In this edition, towards the goal of having some interesting data in the `address_table` as well, below is a more advanced example illustrating how the [Insert.values()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.values) method may be used explicitly while at the same time including the additional VALUES generated from the parameters. A [scalar subquery](https://docs.sqlalchemy.org/en/20/glossary.html#term-scalar-subquery) is constructed, making use of the [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) construct introduced in the next section, and the parameters used in the subquery are set up using an explicit bound parameter name, established using the [bindparam()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.bindparam) construct.

This is some slightly **deeper** alchemy just so that we can add related rows without fetching the primary key identifiers from the `user_table` operation into the applicatoin. Most Alchemists will simply use the ORM which takes care of things like this for us.

In [40]:
from sqlalchemy import select, bindparam
scalar_subq = (
    select(user_table.c.id)
    .where(user_table.c.name == bindparam("username"))
    .scalar_subquery()
)

with engine.connect() as conn:
    result = conn.execute(
        insert(address_table).values(user_id=scalar_subq),
        [
            {
                "username": "spongebob",
                "email_address": "spongebob@sqlalchemy.org",
            },
            {"username": "sandy", "email_address": "sandy@sqlalchemy.org"},
            {"username": "sandy", "email_address": "sandy@squirrelpower.org"},
        ],
    )
    conn.commit()

2026-01-23 08:52:19,927 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:19,928 INFO sqlalchemy.engine.Engine INSERT INTO address (user_id, email_address) VALUES ((SELECT user_account.id 
FROM user_account 
WHERE user_account.name = ?), ?)
2026-01-23 08:52:19,929 INFO sqlalchemy.engine.Engine [generated in 0.00224s] [('spongebob', 'spongebob@sqlalchemy.org'), ('sandy', 'sandy@sqlalchemy.org'), ('sandy', 'sandy@squirrelpower.org')]
2026-01-23 08:52:19,930 INFO sqlalchemy.engine.Engine COMMIT


With that, we have some more interesting data in our tables that we will make use of in the upcoming sections.

### Tip

A true "empty" INSERT that inserts only the "defaults" for a table without including any explicit values at all is generated if we indicate [Insert.values()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.values) with no arguments; not every database backend supports this, but here's what SQLite produces:

In [41]:
print(insert(user_table).values().compile(engine))

INSERT INTO user_account DEFAULT VALUES


## INSERT ... RETURNING

The RETURNING clause for supported backends is used automatically in order to retrieve the last insreted primary key value as well as the values for server defaults. However the RETURNING clause may also be specified explicitly using the [Insert.returning()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.returning) method; in this case, the [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) object that's returned when the statement is executed has rows which can be fetched:

In [42]:
insert_stmt = insert(address_table).returning(
    address_table.c.id,
    address_table.c.email_address
)
print(insert_stmt)

INSERT INTO address (id, user_id, email_address) VALUES (:id, :user_id, :email_address) RETURNING address.id, address.email_address


It can also be combined with [Insert.from_select()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.from_select), as in the example below that builds upon the example stated in [INSERT...FROM SELECT](https://docs.sqlalchemy.org/en/20/tutorial/data_insert.html#tutorial-insert-from-select)

In [43]:
select_stmt = select(user_table.c.id, user_table.c.name + "@aol.com")
insert_stmt = insert(address_table).from_select(
    ["user_id", "email_address"],
    select_stmt
)

print(insert_stmt.returning(address_table.c.id, address_table.c.email_address))

INSERT INTO address (user_id, email_address) SELECT user_account.id, user_account.name || :name_1 AS anon_1 
FROM user_account RETURNING address.id, address.email_address


### Tip

The RETURNING feature is also supported by UPDATE and DELETE statements, which will be introduced later in this tutorial.

For INSERT statements, the RETURNING feature may be used both for single-row statements as well as for statemetns that INSERT multiple rows at once. Support for multiple-row INSERT with RETRUNING is dialect specific, however is supported for all dialetcs that are included in SQLAlchemy which support RETURNING. See the section ["Insert Many Values" Behavior for INSERT statements](https://docs.sqlalchemy.org/en/20/core/connections.html#engine-insertmanyvalues) for background on this feature.

### See also

Bulk INSERT with or without RETURNING is also supported by the ORM. See [ORM Bulk INSERT Statements](https://docs.sqlalchemy.org/en/20/orm/queryguide/dml.html#orm-queryguide-bulk-insert) for reference documentation.


## INSERT ... FROM SELECT

A less used feature of [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert), but here for completeness, the [Insert](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert) construct can compose an INSERT that gets rows directly from a SELECT using the [Insert.from_selec()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.Insert.from_select) method. This method accepts a [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) construct, which is discussed in the next section, along with a list of column names to be targeted in the actual INSERT. In the example below, rows are added to the `address` talbe which are derived from rows in the `user_account` talbe, giving each user a free email address at `aol.com`.:

In [44]:
select_stmt = select(user_table.c.id, user_table.c.name + "@aol.com")
insert_stmt = insert(address_table).from_select(
    ["user_id", "email_address"], select_stmt
)

print(insert_stmt)

INSERT INTO address (user_id, email_address) SELECT user_account.id, user_account.name || :name_1 AS anon_1 
FROM user_account


This construct is used when one wants to ocpy data from some other part of the database directly into a new set of rows, without actually fetching and re-sending data from the client.

# Using SELECT Statements

For both Core and ORM, the [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) function generates a [Select](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.Select) construct which is used for all SELECT queries. Passed to methods like [Connection.execute()](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Connection.execute) in Core and [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) in ORM, a SELECT statement is emitted in the current transaction and the result rows available via the returned [Result](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Result) object.

## ORM Readers 
The content here applies equally well to both Core and ORM use and basic ORM variant use cases are mentioned here. However, there are a lot more ORM-specific features available as well; these are documented at [ORM Querying Guide](https://docs.sqlalchemy.org/en/20/orm/queryguide/index.html)


# The select() SQL Expression Constuct

The [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) construct builds up a statement in the same way as that of [insert()](https://docs.sqlalchemy.org/en/20/core/dml.html#sqlalchemy.sql.expression.insert), using a [generative](https://docs.sqlalchemy.org/en/20/glossary.html#term-generative) approach where each method builds more state onto the object. Like the other SQL constructs, it can be stringified in place:

In [45]:
from sqlalchemy import select
stmt = select(user_table).where(user_table.c.name == "spongebob")
print(stmt)

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1


Also in the same manner as all other statement-level SQL constructs, to actually run the statement we pass it to an execution method. Since a SELECT statement returns rows we can always iterate the result object to get [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects back:

In [46]:
with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row)

2026-01-23 08:52:20,018 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:20,019 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2026-01-23 08:52:20,020 INFO sqlalchemy.engine.Engine [generated in 0.00202s] ('spongebob',)
(1, 'spongebob', 'Spongebob Squarepants')
2026-01-23 08:52:20,021 INFO sqlalchemy.engine.Engine ROLLBACK


When using the ORM, particularly with a [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) constuct that's composed against ORM entities, we will want to execute it using the [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method on the [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session); using this approach, we continue to get [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects from the result, however these rows are now capable of including complete entities, such as instances of the `User` class, as individual elements within each row:

In [47]:
stmt = select(User).where(User.name == "spongebob")
with Session(engine) as session:
    for row in session.execute(stmt):
        print(row)

2026-01-23 08:52:20,034 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:20,035 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2026-01-23 08:52:20,036 INFO sqlalchemy.engine.Engine [generated in 0.00070s] ('spongebob',)
(User(id=1, name='spongebob', fullname='Spongebob Squarepants'),)
2026-01-23 08:52:20,038 INFO sqlalchemy.engine.Engine ROLLBACK


## select() from a Table vs. ORM class

While the SQL generated in these examples looks the same whether we invoke `select(user_table)` or `select(User)`, in the more general case they do not necessarily render the same thing, as an ORM-mapped class may be mapped to other kinds of "selectables" besides tables. The `select()` that's against an ORM entity also indicates that ORM-mapped instances should be returned in a result, which is not the case when SELECTing form a [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) object. 

# Setting the COLUMNS and FROM clause

The [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) function accepts positional elements representing any number of [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) and/or [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) expressions, as well as a wide range of compatible objects, which are resolved into a list of SQL expressions to be SELECTed from that will be returned as columns in the result set. These elements also serve in simpler cases to create the FROM clause, which is inferred from the columns and table-like expressions passed:

In [48]:
print(select(user_table))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account


To SELECT from individual columns using a Core approach, [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) objects are accessed from the [Table.c](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table.c) accessor and can be sent directly; the FROM clause will be inferred as the set of all [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) and other [FromClause](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.FromClause) objects that are represented by those columns.

In [49]:
print(select(user_table.c.name, user_table.c.fullname))

SELECT user_account.name, user_account.fullname 
FROM user_account


Alternatively, when using the [FromClause.c](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.FromClause.c) collection of any [FromClause](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.FromClause) such as [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table), multiple columns may be specified for a [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select) by using a tuple of sring names:


In [50]:
print(select(user_table.c["name", "fullname"]))

SELECT user_account.name, user_account.fullname 
FROM user_account


In [51]:
print(select(user_table.c[("name", "fullname")]))

SELECT user_account.name, user_account.fullname 
FROM user_account


# Selecting ORM Entities and Columns

ORM entities, such as our `User` class as well as the column-mapped attributes upon it such as `User.name`, also participate in the SQL Expression Language system representing tables and columns. Below illustrates an example of SELECTing from the `User` entity, which ultimately renders in the same way as if we had used `user_table` directly:

In [52]:
print(select(User))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account


When executing a statement like the above using the ORM [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute) method, there is an important difference when we select from a full entity such as `User`, as opposed to `user_table`, which is that the **entity itself is returned as a single element within each row**. That is, when we fetch rows from the above statement, s there is only the `User` entity in the list of things to fetch, we get back [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) objects that have only one element, which contain instances of the `User` class:

In [53]:
row = session.execute(select(User)).first()

2026-01-23 08:52:20,108 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:20,111 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2026-01-23 08:52:20,112 INFO sqlalchemy.engine.Engine [generated in 0.00152s] ()


In [54]:
row

(User(id=1, name='spongebob', fullname='Spongebob Squarepants'),)

The above [Row](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.Row) has just one element, representing the `User` entity.

In [55]:
row[0]

User(id=1, name='spongebob', fullname='Spongebob Squarepants')

A highly recommended convenience method of achieving the same result as above is to use the [Session.scalars()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.scalars) method to execute the statement directly; this method will return a [ScalarResult](https://docs.sqlalchemy.org/en/20/core/connections.html#sqlalchemy.engine.ScalarResult) object that delivers the first "columns" of each row at once, in this case, instances of the `User` class:

In [56]:
user = session.scalars(select(User)).first()

2026-01-23 08:52:20,152 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account
2026-01-23 08:52:20,154 INFO sqlalchemy.engine.Engine [cached since 0.04362s ago] ()


In [57]:
user

User(id=1, name='spongebob', fullname='Spongebob Squarepants')

Alternatively, we can select individual columns of an ORM entity as distinct elements within result rows, by using the class-bound attributes; when these are passed to a construct such as [select()](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.select), they are resolved into the [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) or other SQL expression represented by each attribute. 

In [58]:
print(select(User.name, User.fullname))

SELECT user_account.name, user_account.fullname 
FROM user_account


When we invoke _this_ statement using [Session.execute()](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session.execute), we now receive rows that have individual elements per value, each corresponding to a separate column or other SQL expression:

In [59]:
row = session.execute(select(User.name, User.fullname)).first()

2026-01-23 08:52:20,202 INFO sqlalchemy.engine.Engine SELECT user_account.name, user_account.fullname 
FROM user_account
2026-01-23 08:52:20,204 INFO sqlalchemy.engine.Engine [generated in 0.00204s] ()


In [60]:
row

('spongebob', 'Spongebob Squarepants')

The approaches can also be mixed, as below where we SELECT the `name` attribute of the `User` entity as the first element of the row, and combine it with full `Address` entities in the second element:

In [61]:
session.execute(
    select(User.name, Address).where(User.id == Address.user_id).order_by(Address.id)
).all()


2026-01-23 08:52:20,248 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.id, address.user_id, address.email_address 
FROM user_account, address 
WHERE user_account.id = address.user_id ORDER BY address.id
2026-01-23 08:52:20,249 INFO sqlalchemy.engine.Engine [generated in 0.00121s] ()


[('spongebob', Address(id=1, user_id=1, email_address='spongebob@sqlalchemy.org')),
 ('sandy', Address(id=2, user_id=2, email_address='sandy@sqlalchemy.org')),
 ('sandy', Address(id=3, user_id=2, email_address='sandy@squirrelpower.org'))]

Approaches towards selecting ORM entities and columns as well as common methods for converting rows are discussed further at [Selecting ORM Entites and Attributes](https://docs.sqlalchemy.org/en/20/orm/queryguide/select.html#orm-queryguide-select-columns)

See also: [Selecting ORM Entites and Attritubes](https://docs.sqlalchemy.org/en/20/orm/queryguide/select.html#orm-queryguide-select-columns) in the [ORM Querying Guide](https://docs.sqlalchemy.org/en/20/orm/queryguide/index.html)


# Selecting from Labeled SQL Expressions

The [ColumnElement.label()](https://docs.sqlalchemy.org/en/20/core/sqlelement.html#sqlalchemy.sql.expression.ColumnElement.label) method as well as the same named method available on ORM attributes provides a SQL label of a column or expression, allowing it to have a specific name in a result set. This can be helpful when referring to arbitrary SQL expressions in a result row by name:

In [62]:
from sqlalchemy import func, cast
stmt = select(
    ("Username: " + user_table.c.name).label("username"),
).order_by(user_table.c.name)


with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(f"{row.username}")

2026-01-23 08:52:20,263 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:52:20,264 INFO sqlalchemy.engine.Engine SELECT ? || user_account.name AS username 
FROM user_account ORDER BY user_account.name
2026-01-23 08:52:20,264 INFO sqlalchemy.engine.Engine [generated in 0.00166s] ('Username: ',)
Username: patrick
Username: sandy
Username: spongebob
2026-01-23 08:52:20,266 INFO sqlalchemy.engine.Engine ROLLBACK


### See also

[Ordering or Grouping by a Label](https://docs.sqlalchemy.org/en/20/tutorial/data_select.html#tutorial-order-by-label) - the label names we create may also be referenced in the ORDER BY or GROUP BY clause of the [Select](https://docs.sqlalchemy.org/en/20/core/selectable.html#sqlalchemy.sql.expression.Select)

In [63]:
from sqlalchemy import text
stmt = select(text("'some phrase'"), user_table.c.name).order_by(user_table.c.name)
with engine.connect() as conn:
    print(conn.execute(stmt).all())

2026-01-23 08:55:21,523 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:55:21,524 INFO sqlalchemy.engine.Engine SELECT 'some phrase', user_account.name 
FROM user_account ORDER BY user_account.name
2026-01-23 08:55:21,525 INFO sqlalchemy.engine.Engine [generated in 0.00224s] ()
[('some phrase', 'patrick'), ('some phrase', 'sandy'), ('some phrase', 'spongebob')]
2026-01-23 08:55:21,526 INFO sqlalchemy.engine.Engine ROLLBACK


In [64]:
from sqlalchemy import literal_column
stmt = select(literal_column("'some phrase'").label("p"), user_table.c.name).order_by(
    user_table.c.name
)
with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(f"{row.p}, {row.name}")

2026-01-23 08:58:12,292 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 08:58:12,293 INFO sqlalchemy.engine.Engine SELECT 'some phrase' AS p, user_account.name 
FROM user_account ORDER BY user_account.name
2026-01-23 08:58:12,293 INFO sqlalchemy.engine.Engine [generated in 0.00162s] ()
some phrase, patrick
some phrase, sandy
some phrase, spongebob
2026-01-23 08:58:12,294 INFO sqlalchemy.engine.Engine ROLLBACK


In [65]:
print(user_table.c.name == "squidward")


user_account.name = :name_1


In [66]:
print(address_table.c.user_id > 10)


address.user_id > :user_id_1


In [67]:
print(select(user_table).where(user_table.c.name == "squidward"))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1


In [68]:
print(
    select(address_table.c.email_address)
    .where(user_table.c.id == "squidward")
    .where(address_table.c.user_id == user_table.c.id)
)

SELECT address.email_address 
FROM address, user_account 
WHERE user_account.id = :id_1 AND address.user_id = user_account.id


In [69]:
print(
    select(address_table.c.email_address).where(
        user_table.c.name == "squidward",
        address_table.c.user_id == user_table.c.id,
    )
)

SELECT address.email_address 
FROM address, user_account 
WHERE user_account.name = :name_1 AND address.user_id = user_account.id


In [70]:
from sqlalchemy import and_, or_
print(
    select(Address.email_address).where(
        and_(
            or_(User.name == "squidward", User.name == "sandy"),
            Address.user_id == User.id,
        )
    )
)

SELECT address.email_address 
FROM address, user_account 
WHERE (user_account.name = :name_1 OR user_account.name = :name_2) AND address.user_id = user_account.id


In [71]:
print(select(User).filter_by(name="spongebob", fullname="Spongebob Squarepants"))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = :name_1 AND user_account.fullname = :fullname_1


In [72]:
print(select(user_table.c.name))

SELECT user_account.name 
FROM user_account


In [73]:
print(select(user_table.c.name, address_table.c.email_address))

SELECT user_account.name, address.email_address 
FROM user_account, address


In [74]:
print(
    select(user_table.c.name, address_table.c.email_address).join_from(
        user_table, address_table
    )
)

SELECT user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [75]:
print(select(user_table.c.name, address_table.c.email_address).join(address_table))

SELECT user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [76]:
print(select(address_table.c.email_address).select_from(user_table).join(address_table))

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [77]:
from sqlalchemy import func
print(select(func.count("*")).select_from(user_table))

SELECT count(:count_2) AS count_1 
FROM user_account


In [78]:
print(
    select(address_table.c.email_address)
    .select_from(user_table)
    .join(address_table, user_table.c.id == address_table.c.user_id)
)

SELECT address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id


In [79]:
print(select(user_table).join(address_table, isouter=True))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account LEFT OUTER JOIN address ON user_account.id = address.user_id


In [80]:
print(select(user_table).join(address_table, full=True))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account FULL OUTER JOIN address ON user_account.id = address.user_id


In [81]:
print(select(user_table).order_by(user_table.c.name))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account ORDER BY user_account.name


In [82]:
print(select(User).order_by(User.fullname.desc()))

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account ORDER BY user_account.fullname DESC


In [83]:
from sqlalchemy import func
count_fn = func.count(user_table.c.id)
print(count_fn)

count(user_account.id)


In [84]:
with engine.connect() as conn:
    result = conn.execute(
        select(User.name, func.count(Address.id).label("count"))
        .join(Address)
        .group_by(User.name)
        .having(func.count(Address.id) > 1)
    )
    print(result.all())

2026-01-23 09:24:58,623 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 09:24:58,626 INFO sqlalchemy.engine.Engine SELECT user_account.name, count(address.id) AS count 
FROM user_account JOIN address ON user_account.id = address.user_id GROUP BY user_account.name 
HAVING count(address.id) > ?
2026-01-23 09:24:58,627 INFO sqlalchemy.engine.Engine [generated in 0.00436s] (1,)
[('sandy', 2)]
2026-01-23 09:24:58,629 INFO sqlalchemy.engine.Engine ROLLBACK


In [85]:
from sqlalchemy import func, desc
stmt = (
    select(Address.user_id, func.count(Address.id).label("num_addresses"))
    .group_by("user_id")
    .order_by("user_id", desc("num_addresses"))
)
print(stmt)

SELECT address.user_id, count(address.id) AS num_addresses 
FROM address GROUP BY address.user_id ORDER BY address.user_id, num_addresses DESC


In [90]:
user_alias_1 = user_table.alias()
user_alias_2 = user_table.alias()
stmt = select(user_alias_1.c.name, user_alias_2.c.name).join_from(
    user_alias_1, user_alias_2, user_alias_1.c.id > user_alias_2.c.id
)
print(
    select(user_alias_1.c.name, user_alias_2.c.name).join_from(
        user_alias_1, user_alias_2, user_alias_1.c.id > user_alias_2.c.id
    )
)

with engine.connect() as conn:
    result = conn.execute(stmt)
    for row in result:
        print(row, row[0], row[1])

SELECT user_account_1.name, user_account_2.name AS name_1 
FROM user_account AS user_account_1 JOIN user_account AS user_account_2 ON user_account_1.id > user_account_2.id
2026-01-23 09:33:45,106 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 09:33:45,109 INFO sqlalchemy.engine.Engine SELECT user_account_1.name, user_account_2.name AS name_1 
FROM user_account AS user_account_1 JOIN user_account AS user_account_2 ON user_account_1.id > user_account_2.id
2026-01-23 09:33:45,112 INFO sqlalchemy.engine.Engine [cached since 98.3s ago] ()
('sandy', 'spongebob') sandy spongebob
('patrick', 'spongebob') patrick spongebob
('patrick', 'sandy') patrick sandy
2026-01-23 09:33:45,117 INFO sqlalchemy.engine.Engine ROLLBACK


In [91]:
from sqlalchemy.orm import aliased
address_alias_1 = aliased(Address)
address_alias_2 = aliased(Address)
print(
    select(User)
    .join_from(User, address_alias_1)
    .where(address_alias_1.email_address == "patrick@aol.com")
    .join_from(User, address_alias_2)
    .where(address_alias_2.email_address == "patrick@gmail.com")
)

SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account JOIN address AS address_1 ON user_account.id = address_1.user_id JOIN address AS address_2 ON user_account.id = address_2.user_id 
WHERE address_1.email_address = :email_address_1 AND address_2.email_address = :email_address_2


In [92]:
subq = (
    select(func.count(address_table.c.id).label("count"), address_table.c.user_id)
    .group_by(address_table.c.user_id)
    .subquery()
    )


In [93]:
print(subq)

SELECT count(address.id) AS count, address.user_id 
FROM address GROUP BY address.user_id


In [94]:
print(select(subq.c.user_id, subq.c.count))

SELECT anon_1.user_id, anon_1.count 
FROM (SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id) AS anon_1


In [95]:
stmt = select(user_table.c.name, user_table.c.fullname, subq.c.count).join_from(
    user_table, subq
)

In [96]:
print(stmt)

SELECT user_account.name, user_account.fullname, anon_1.count 
FROM user_account JOIN (SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id) AS anon_1 ON user_account.id = anon_1.user_id


In [97]:
subq = (
    select(func.count(address_table.c.id).label("count"), address_table.c.user_id)
    .group_by(address_table.c.user_id)
    .cte()
)

stmt = select(user_table.c.name, user_table.c.fullname, subq.c.count).join_from(
    user_table, subq
)

print(stmt)

WITH anon_1 AS 
(SELECT count(address.id) AS count, address.user_id AS user_id 
FROM address GROUP BY address.user_id)
 SELECT user_account.name, user_account.fullname, anon_1.count 
FROM user_account JOIN anon_1 ON user_account.id = anon_1.user_id


In [98]:
subq = select(Address).where(~Address.email_address.like("%@aol.com")).subquery()
address_subq = aliased(Address, subq)
stmt = (
    select(User, address_subq)
    .join_from(User, address_subq)
    .order_by(User.id, address_subq.id)
)

with Session(engine) as session:
    for user, address in session.execute(stmt):
        print(f"{user} {address}")

2026-01-23 09:49:28,932 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 09:49:28,938 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, anon_1.id AS id_1, anon_1.user_id, anon_1.email_address 
FROM user_account JOIN (SELECT address.id AS id, address.user_id AS user_id, address.email_address AS email_address 
FROM address 
WHERE address.email_address NOT LIKE ?) AS anon_1 ON user_account.id = anon_1.user_id ORDER BY user_account.id, anon_1.id
2026-01-23 09:49:28,943 INFO sqlalchemy.engine.Engine [generated in 0.00527s] ('%@aol.com',)
User(id=1, name='spongebob', fullname='Spongebob Squarepants') Address(id=1, user_id=1, email_address='spongebob@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=2, user_id=2, email_address='sandy@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=3, user_id=2, email_address='sandy@squirrelpower.org')
2026-01-23 09:49:28,946 INFO sqlalchemy.engine.

In [99]:
cte_obj = select(Address).where(~Address.email_address.like("%@aol.com")).cte()
address_cte = aliased(Address, cte_obj)
stmt = (
    select(User, address_cte)
    .join_from(User, address_cte)
    .order_by(User.id, address_cte.id)
)

with Session(engine) as session:
    for user, address in session.execute(stmt):
        print(f"{user} {address}")

2026-01-23 09:52:10,576 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 09:52:10,582 INFO sqlalchemy.engine.Engine WITH anon_1 AS 
(SELECT address.id AS id, address.user_id AS user_id, address.email_address AS email_address 
FROM address 
WHERE address.email_address NOT LIKE ?)
 SELECT user_account.id, user_account.name, user_account.fullname, anon_1.id AS id_1, anon_1.user_id, anon_1.email_address 
FROM user_account JOIN anon_1 ON user_account.id = anon_1.user_id ORDER BY user_account.id, anon_1.id
2026-01-23 09:52:10,583 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ('%@aol.com',)
User(id=1, name='spongebob', fullname='Spongebob Squarepants') Address(id=1, user_id=1, email_address='spongebob@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=2, user_id=2, email_address='sandy@sqlalchemy.org')
User(id=2, name='sandy', fullname='Sandy Cheeks') Address(id=3, user_id=2, email_address='sandy@squirrelpower.org')
2026-01-23 09:52:10,586 INFO sqla

In [100]:
subq = (
    select(func.count(address_table.c.id))
    .where(user_table.c.id == address_table.c.user_id)
    .scalar_subquery()
)

print(subq)

(SELECT count(address.id) AS count_1 
FROM address, user_account 
WHERE user_account.id = address.user_id)


In [101]:
print(subq == 5)

(SELECT count(address.id) AS count_1 
FROM address, user_account 
WHERE user_account.id = address.user_id) = :param_1


In [102]:
stmt = select(user_table.c.name, subq.label("address_count"))
print(stmt)

SELECT user_account.name, (SELECT count(address.id) AS count_1 
FROM address 
WHERE user_account.id = address.user_id) AS address_count 
FROM user_account


In [105]:
subq = (
    select(func.count(address_table.c.id))
    .where(user_table.c.id == address_table.c.user_id)
    .scalar_subquery()
    .correlate(user_table)
)

In [106]:
with engine.connect() as conn:
    result = conn.execute(
        select(
            user_table.c.name,
            address_table.c.email_address,
            subq.label("address_count"),
        )
        .join_from(user_table, address_table)
        .order_by(user_table.c.id, address_table.c.id)
    )

    print(result.all())

2026-01-23 10:05:30,874 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 10:05:30,876 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.email_address, (SELECT count(address.id) AS count_1 
FROM address 
WHERE user_account.id = address.user_id) AS address_count 
FROM user_account JOIN address ON user_account.id = address.user_id ORDER BY user_account.id, address.id
2026-01-23 10:05:30,877 INFO sqlalchemy.engine.Engine [generated in 0.00366s] ()
[('spongebob', 'spongebob@sqlalchemy.org', 1), ('sandy', 'sandy@sqlalchemy.org', 2), ('sandy', 'sandy@squirrelpower.org', 2)]
2026-01-23 10:05:30,880 INFO sqlalchemy.engine.Engine ROLLBACK
